<p align="center">
  <img src="https://www.copernicuslac-chile.eu/wp-content/uploads/2024/05/logo_copernicuslac_chile.svg" alt="Copernicus LAC Chile" width="1600"/>
</p>

# 00 — Verificación del entorno

Taller **CopernicusLAC Chile — IDE Uruguay**

Ejecuta este notebook al inicio del taller para revisar Python, las librerías definidas en `environment.yml` y operaciones básicas con los datos versionados del repositorio.

Una dependencia o una operación que falla se informa como **ERROR**. Si no hay un archivo opcional para probar, se informa como **AVISO** y eso por sí solo no impide usar el entorno.

## 1. Python y kernel

El entorno definido para el taller es `2026coplac`, con Python 3.11.

In [ ]:
import importlib
import importlib.util
import platform
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
print("Sistema operativo:", platform.system(), platform.release())
print("Entorno esperado: 2026coplac (Python 3.11)")

## 2. Dependencias

Se comprueban los paquetes declarados en `environment.yml` y los módulos de Python necesarios para los notebooks.

In [ ]:
DEPENDENCIAS = [
    ("Base", "numpy", "numpy"),
    ("Base", "pandas", "pandas"),
    ("Visualización", "matplotlib", "matplotlib"),
    ("Visualización", "matplotlib_inline", "matplotlib-inline"),
    ("Visualización", "seaborn", "seaborn"),
    ("Geoespacial", "geopandas", "geopandas"),
    ("Geoespacial", "shapely", "shapely"),
    ("Geoespacial", "pyproj", "pyproj"),
    ("Geoespacial", "fiona", "fiona"),
    ("Geoespacial", "pyogrio", "pyogrio"),
    ("Geoespacial", "rasterio", "rasterio"),
    ("Geoespacial", "rioxarray", "rioxarray"),
    ("Geoespacial", "affine", "affine"),
    ("Geoespacial", "cartopy", "cartopy"),
    ("Datos", "xarray", "xarray"),
    ("Datos", "netCDF4", "netcdf4"),
    ("Datos", "scipy", "scipy"),
    ("Jupyter", "IPython", "ipython"),
    ("Jupyter", "ipywidgets", "ipywidgets"),
    ("Jupyter", "ipympl", "ipympl"),
    ("Jupyter", "ipykernel", "ipykernel"),
    ("Jupyter", "jupyter_client", "jupyter_client"),
    ("Jupyter", "zmq", "pyzmq"),
    ("Jupyter", "tornado", "tornado"),
    ("Jupyter", "traitlets", "traitlets"),
    ("Jupyter", "nbclient", "nbclient"),
    ("Jupyter", "nbconvert", "nbconvert"),
    ("Jupyter", "jupyterlab", "jupyterlab"),
    ("Utilidades", "requests", "requests"),
    ("Utilidades", "pooch", "pooch"),
    ("Utilidades", "unidecode", "unidecode"),
    ("Utilidades", "gdown", "gdown"),
    ("Utilidades", "openpyxl", "openpyxl"),
    ("Utilidades", "folium", "folium"),
]

resultados_dependencias = []
modulos = {}

for grupo, nombre_import, nombre_paquete in DEPENDENCIAS:
    disponible = importlib.util.find_spec(nombre_import) is not None
    version = "—"
    detalle = ""
    if disponible:
        try:
            modulo = importlib.import_module(nombre_import)
            modulos[nombre_import] = modulo
            version = getattr(modulo, "__version__", "instalado")
            estado = "OK"
        except Exception as exc:
            estado = "ERROR DE IMPORTACIÓN"
            detalle = f"{type(exc).__name__}: {exc}"
    else:
        estado = "FALTA"
        detalle = f"Instalar paquete: {nombre_paquete}"

    resultados_dependencias.append({
        "grupo": grupo,
        "paquete": nombre_paquete,
        "import": nombre_import,
        "versión": version,
        "estado": estado,
        "detalle": detalle,
    })

version_python_ok = sys.version_info[:2] == (3, 11)
resultados_dependencias.append({
    "grupo": "Entorno",
    "paquete": "python=3.11",
    "import": "sys",
    "versión": sys.version.split()[0],
    "estado": "OK" if version_python_ok else "ERROR",
    "detalle": "" if version_python_ok else "El entorno del taller requiere Python 3.11.",
})

if "pandas" in modulos and "IPython" in modulos:
    from IPython.display import display
    display(modulos["pandas"].DataFrame(resultados_dependencias))
else:
    for fila in resultados_dependencias:
        print(f"{fila['estado']:22s} | {fila['grupo']:14s} | {fila['paquete']} {fila['detalle']}")

faltantes = [fila for fila in resultados_dependencias if fila["estado"] != "OK"]
print(f"\nDependencias con problemas: {len(faltantes)} de {len(resultados_dependencias)}")
if faltantes:
    for fila in faltantes:
        print(f" - [{fila['grupo']}] {fila['paquete']}: {fila['estado']} {fila['detalle']}")

## 3. Estructura y datos del repositorio

La búsqueda identifica la raíz a partir de `environment.yml` y `data/`, tanto si Jupyter se inició desde la raíz como desde `notebooks/`. Solo se inspecciona `data/`, que contiene los insumos versionados del taller.

In [ ]:
def encontrar_base_proyecto():
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "environment.yml").is_file() and (candidato / "data").is_dir():
            return candidato
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio (se requieren environment.yml y data/). "
        "Inicia Jupyter desde el repositorio o desde notebooks/."
    )


BASE_DIR = encontrar_base_proyecto()
DATA_DIR = BASE_DIR / "data"

carpetas_requeridas = ["data", "notebooks", "scripts"]
print("Raíz del proyecto:", BASE_DIR)
for nombre in carpetas_requeridas:
    ruta = BASE_DIR / nombre
    print(f"{'OK' if ruta.is_dir() else 'NO PRESENTE':12s} {nombre}/")

SHAPES_AU_DIR = DATA_DIR / "shapes_AU"
SHAPES_CYU_DIR = DATA_DIR / "shapes_CyU"
RASTERS_AU_DIR = DATA_DIR / "rasters_AU"
RASTERS_CYU_DIR = DATA_DIR / "rasters_CyU"

for nombre, ruta in [
    ("vectores Atlas Urbano", SHAPES_AU_DIR),
    ("vectores Cobertura y Uso", SHAPES_CYU_DIR),
    ("rasters Atlas Urbano", RASTERS_AU_DIR),
    ("rasters Cobertura y Uso", RASTERS_CYU_DIR),
]:
    print(f"{'OK' if ruta.is_dir() else 'NO PRESENTE':12s} {nombre}: {ruta.relative_to(BASE_DIR)}")

def buscar_archivos(carpeta, extensiones):
    if not carpeta.is_dir():
        return []
    extensiones = {extension.casefold() for extension in extensiones}
    return sorted(
        ruta for ruta in carpeta.rglob("*")
        if ruta.is_file() and ruta.suffix.casefold() in extensiones
    )


vectores_au = buscar_archivos(SHAPES_AU_DIR, {".shp", ".gpkg", ".geojson"})
vectores_cyu = buscar_archivos(SHAPES_CYU_DIR, {".shp", ".gpkg", ".geojson"})
rasters_au = buscar_archivos(RASTERS_AU_DIR, {".tif", ".tiff", ".vrt"})
rasters_cyu = buscar_archivos(RASTERS_CYU_DIR, {".tif", ".tiff", ".vrt"})

for etiqueta, archivos in [
    ("Vectores Atlas Urbano", vectores_au),
    ("Vectores Cobertura y Uso", vectores_cyu),
    ("Rasters Atlas Urbano", rasters_au),
    ("Rasters Cobertura y Uso", rasters_cyu),
]:
    print(f"{etiqueta}: {len(archivos)} archivo(s)")
    for archivo in archivos[:3]:
        print("  -", archivo.relative_to(BASE_DIR))
    if len(archivos) > 3:
        print(f"  ... y {len(archivos) - 3} más")

## 4. Pruebas funcionales

Además de comprobar que los módulos importan, esta sección prueba reproyección de geometrías, lectura de las capas y rasters disponibles, NetCDF, Cartopy, widgets y Folium. Los archivos de datos ausentes se informan como avisos.

In [ ]:
pruebas = []

def registrar(servicio, prueba, estado, detalle=""):
    pruebas.append({
        "servicio": servicio,
        "prueba": prueba,
        "estado": estado,
        "detalle": detalle,
    })
    sufijo = f" — {detalle}" if detalle else ""
    print(f"{estado:7s} | {servicio:22s} | {prueba}{sufijo}")


# GeoPandas y proyección local de Uruguay (UTM 21S).
if all(nombre in modulos for nombre in ("geopandas", "shapely")):
    try:
        from shapely.geometry import Point
        gpd = modulos["geopandas"]
        punto = gpd.GeoDataFrame({"id": [1]}, geometry=[Point(-56.16, -34.90)], crs="EPSG:4326")
        reproyectado = punto.to_crs("EPSG:32721")
        registrar("Común", "GeoPandas y reproyección EPSG:32721", "OK", f"x={reproyectado.geometry.iloc[0].x:.0f}, y={reproyectado.geometry.iloc[0].y:.0f}")
    except Exception as exc:
        registrar("Común", "GeoPandas y reproyección EPSG:32721", "ERROR", f"{type(exc).__name__}: {exc}")
else:
    registrar("Común", "GeoPandas y reproyección EPSG:32721", "OMITIDO", "falta geopandas o shapely")


# Lectura de capas vectoriales presentes en los datos versionados.
if "geopandas" in modulos:
    gpd = modulos["geopandas"]
    for servicio, archivos in [("Atlas Urbano", vectores_au), ("Cobertura y Uso", vectores_cyu)]:
        if not archivos:
            registrar(servicio, "Lectura de capa vectorial", "AVISO", "no hay una capa vectorial disponible en data/")
            continue
        try:
            capa = gpd.read_file(archivos[0])
            registrar(servicio, "Lectura de capa vectorial", "OK", f"{archivos[0].name}; {len(capa)} registros; CRS={capa.crs}")
        except Exception as exc:
            registrar(servicio, "Lectura de capa vectorial", "ERROR", f"{archivos[0].name}: {type(exc).__name__}: {exc}")
else:
    registrar("Datos vectoriales", "Lectura de capas", "OMITIDO", "geopandas ausente")


# Rasterio y rioxarray leen una ventana pequeña para limitar memoria.
if "rasterio" in modulos:
    rasterio = modulos["rasterio"]
    for servicio, archivos in [("Atlas Urbano", rasters_au), ("Cobertura y Uso", rasters_cyu)]:
        if not archivos:
            registrar(servicio, "Lectura de raster", "AVISO", "no hay un raster disponible en data/")
            continue
        try:
            with rasterio.open(archivos[0]) as src:
                ventana = src.read(1, window=((0, min(8, src.height)), (0, min(8, src.width))))
                detalle = f"{archivos[0].name}; {src.width}x{src.height}; CRS={src.crs}; dtype={ventana.dtype}"
            registrar(servicio, "Lectura de raster", "OK", detalle)
        except Exception as exc:
            registrar(servicio, "Lectura de raster", "ERROR", f"{archivos[0].name}: {type(exc).__name__}: {exc}")
else:
    registrar("Datos raster", "Lectura de rasters", "OMITIDO", "rasterio ausente")

if "rioxarray" in modulos and rasters_au:
    try:
        with modulos["rioxarray"].open_rasterio(rasters_au[0], chunks=None) as raster:
            detalle = f"{rasters_au[0].name}; dimensiones={dict(raster.sizes)}"
        registrar("Atlas Urbano", "Lectura con rioxarray", "OK", detalle)
    except Exception as exc:
        registrar("Atlas Urbano", "Lectura con rioxarray", "ERROR", f"{type(exc).__name__}: {exc}")
elif "rioxarray" not in modulos:
    registrar("Atlas Urbano", "Lectura con rioxarray", "OMITIDO", "rioxarray ausente")
else:
    registrar("Atlas Urbano", "Lectura con rioxarray", "AVISO", "no hay un raster disponible en data/")


# Escribir y volver a leer un NetCDF temporal comprueba el motor netCDF4 sin requerir datos externos.
if all(nombre in modulos for nombre in ("numpy", "xarray", "netCDF4")):
    try:
        import tempfile
        import numpy as np
        xr = modulos["xarray"]
        with tempfile.TemporaryDirectory() as carpeta_temporal:
            archivo_nc = Path(carpeta_temporal) / "prueba_entorno.nc"
            xr.Dataset({"valor": (("y", "x"), np.array([[1, 2], [3, 4]]))}).to_netcdf(archivo_nc, engine="netcdf4")
            with xr.open_dataset(archivo_nc, engine="netcdf4") as ds:
                detalle = f"variables={list(ds.data_vars)}; dimensiones={dict(ds.sizes)}"
        registrar("Datos", "Escritura y lectura NetCDF", "OK", detalle)
    except Exception as exc:
        registrar("Datos", "Escritura y lectura NetCDF", "ERROR", f"{type(exc).__name__}: {exc}")
else:
    registrar("Datos", "Escritura y lectura NetCDF", "OMITIDO", "falta numpy, xarray o netCDF4")


# Cartopy transforma coordenadas sin descargar capas cartográficas externas.
if "cartopy" in modulos:
    try:
        import cartopy.crs as ccrs
        x, y = ccrs.UTM(21).transform_point(-56.16, -34.90, ccrs.PlateCarree())
        registrar("Geoespacial", "Proyección Cartopy UTM 21S", "OK", f"x={x:.0f}, y={y:.0f}")
    except Exception as exc:
        registrar("Geoespacial", "Proyección Cartopy UTM 21S", "ERROR", f"{type(exc).__name__}: {exc}")
else:
    registrar("Geoespacial", "Proyección Cartopy UTM 21S", "OMITIDO", "cartopy ausente")


if "matplotlib" in modulos:
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(5, 2.5))
        ax.plot([0, 1, 2], [0, 1, 0])
        ax.set_title("Prueba Matplotlib — IDE Uruguay")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.tight_layout()
        plt.show()
        registrar("Visualización", "Matplotlib", "OK")
    except Exception as exc:
        registrar("Visualización", "Matplotlib", "ERROR", f"{type(exc).__name__}: {exc}")
else:
    registrar("Visualización", "Matplotlib", "OMITIDO", "matplotlib ausente")

if "ipywidgets" in modulos:
    try:
        _ = modulos["ipywidgets"].Dropdown(options=["Uruguay"], value="Uruguay", description="Test")
        registrar("Jupyter", "ipywidgets", "OK")
    except Exception as exc:
        registrar("Jupyter", "ipywidgets", "ERROR", f"{type(exc).__name__}: {exc}")
else:
    registrar("Jupyter", "ipywidgets", "OMITIDO", "ipywidgets ausente")

if "folium" in modulos:
    try:
        _ = modulos["folium"].Map(location=[-32.5, -55.8], zoom_start=6)
        registrar("Visualización", "Mapa Folium de Uruguay", "OK")
    except Exception as exc:
        registrar("Visualización", "Mapa Folium de Uruguay", "ERROR", f"{type(exc).__name__}: {exc}")
else:
    registrar("Visualización", "Mapa Folium de Uruguay", "OMITIDO", "folium ausente")

## 5. Resultado

El entorno queda listo si todas las dependencias importan y ninguna prueba funcional produce **ERROR**. Los avisos sobre archivos opcionales no bloquean la validación.

In [ ]:
problemas_dependencias = [fila for fila in resultados_dependencias if fila["estado"] != "OK"]
problemas_funcionales = [fila for fila in pruebas if fila["estado"] == "ERROR"]
avisos = [fila for fila in pruebas if fila["estado"] in ("AVISO", "OMITIDO")]

print("=" * 76)
print("RESUMEN — VERIFICACIÓN DEL ENTORNO IDE URUGUAY")
print("=" * 76)
print(f"Dependencias con problemas : {len(problemas_dependencias)}")
print(f"Pruebas funcionales ERROR  : {len(problemas_funcionales)}")
print(f"Avisos / pruebas omitidas   : {len(avisos)}")

if problemas_dependencias:
    print("\nDEPENDENCIAS A CORREGIR:")
    for fila in problemas_dependencias:
        print(f" - [{fila['grupo']}] {fila['paquete']}: {fila['estado']} {fila['detalle']}")

if problemas_funcionales:
    print("\nERRORES FUNCIONALES A CORREGIR:")
    for fila in problemas_funcionales:
        print(f" - [{fila['servicio']}] {fila['prueba']}: {fila['detalle']}")

if avisos:
    print("\nAVISOS (no bloquean por sí solos):")
    for fila in avisos:
        print(f" - [{fila['servicio']}] {fila['prueba']}: {fila['detalle']}")

if not problemas_dependencias and not problemas_funcionales:
    print("\nENTORNO LISTO PARA EL TALLER IDE URUGUAY.")
else:
    print("\nENTORNO NO LISTO. Corrige los problemas indicados antes de continuar.")